#  Python Tutorial — Part 7: Advanced OOP

**Topics Covered:**
- Polymorphism
- Abstract Base Classes (`abc`)
- Class decorators: `@classmethod`, `@staticmethod`, `@property`
- Magic / Dunder methods
- Multiple Inheritance
- Type annotations

---

## 1. Polymorphism

**Polymorphism** is the ability for different classes to respond to the same method call, each in their own way. Think of it like a universal remote: pressing the "play" button works on a DVD player, a music app, and a streaming service — the button is identical, but each device carries out the action according to its own internal logic.

This lets you write code that operates on objects without needing to know their exact type, as long as those objects expose the expected method.

In [ ]:
class Parrot:
    def fly(self):
        return 'Parrot is flying'
    def swim(self):
        return "Parrot can't swim"

class Penguin:
    def fly(self):
        return "Penguin can't fly"
    def swim(self):
        return 'Penguin is swimming'


# Polymorphic function — works with ANY bird-like object
def describe_bird(bird):
    print(bird.fly())
    print(bird.swim())
    print()

describe_bird(Parrot())
describe_bird(Penguin())

In [ ]:
# Works with mixed lists too!
birds = [Parrot(), Penguin(), Parrot()]
for bird in birds:
    print(bird.fly())

---
## 2. Abstract Base Classes (`abc`)

An **abstract class** acts as a shared template that sets out which methods every subclass is required to provide. It specifies the structure without supplying a complete implementation — subclasses fill in the details. Because the abstract class is intentionally incomplete, Python will not allow you to instantiate it directly.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):           # Inherit from ABC to mark as abstract
    @abstractmethod
    def area(self) -> float:
        """Every Shape MUST implement area()."""
        pass

    @abstractmethod
    def perimeter(self) -> float:
        pass

    def describe(self):
        # Non-abstract method — shared by all shapes
        return f'Area={self.area():.2f}, Perimeter={self.perimeter():.2f}'

In [ ]:
# Cannot instantiate abstract class
try:
    s = Shape()
except TypeError as e:
    print(f'Error: {e}')

In [ ]:
# Concrete subclasses MUST implement all abstract methods
class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)


class Circle(Shape):
    import math
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        import math
        return math.pi * self.radius ** 2

    def perimeter(self):
        import math
        return 2 * math.pi * self.radius


shapes = [Rectangle(4, 5), Circle(7)]
for shape in shapes:
    print(type(shape).__name__, '->', shape.describe())

---
## 3. Class Decorators

### `@classmethod`
Bound to the **class**, not the instance. First parameter is `cls` (the class itself).  
Use for **factory methods** and operations on class state.

In [ ]:
from datetime import date

class Person:
    def __init__(self, name, age):
        self.name = name
        self.age  = age

    @classmethod
    def from_birth_year(cls, name, birth_year):
        """Factory: create a Person from birth year instead of age."""
        age = date.today().year - birth_year
        return cls(name, age)   # cls(...) creates an instance of the class

    def __str__(self):
        return f'{self.name} (age {self.age})'


p1 = Person('Alice', 30)
p2 = Person.from_birth_year('Bob', 1995)

print(p1)
print(p2)

### `@staticmethod`
Not bound to instance or class. No `self` or `cls`.  
Use for **utility functions** that logically belong to the class but don't need its state.

In [ ]:
class TemperatureConverter:
    @staticmethod
    def celsius_to_fahrenheit(c):
        return c * 9/5 + 32

    @staticmethod
    def fahrenheit_to_celsius(f):
        return (f - 32) * 5/9


# No need to create an instance
print(TemperatureConverter.celsius_to_fahrenheit(100))  # 212.0
print(TemperatureConverter.fahrenheit_to_celsius(98.6)) # ~37.0

### Comparison: instance, classmethod, staticmethod

In [ ]:
class Person:
    TITLES = ('Dr', 'Mr', 'Mrs', 'Ms')

    def __init__(self, name, surname):
        self.name    = name
        self.surname = surname

    def fullname(self):                       # instance method — needs self
        return f'{self.name} {self.surname}'

    @classmethod
    def titles_starting_with(cls, letter):    # class method — needs cls
        return [t for t in cls.TITLES if t.startswith(letter)]

    @staticmethod
    def is_valid_title(title):                # static method — needs neither
        return title in ('Dr', 'Mr', 'Mrs', 'Ms')


jane = Person('Jane', 'Smith')
print(jane.fullname())                          # instance
print(Person.titles_starting_with('M'))         # classmethod via class
print(jane.titles_starting_with('M'))           # classmethod via instance
print(Person.is_valid_title('Dr'))              # staticmethod
print(Person.is_valid_title('Sir'))             # staticmethod

### `@property`
Lets a **method** be accessed like an **attribute** — no parentheses needed.

In [ ]:
class Person:
    def __init__(self, name, surname):
        self.name    = name
        self.surname = surname

    @property
    def fullname(self):          # getter
        return f'{self.name} {self.surname}'

    @fullname.setter
    def fullname(self, value):   # setter — split on space
        parts = value.split()
        self.name, self.surname = parts[0], parts[-1]

    @fullname.deleter
    def fullname(self):          # deleter
        del self.name
        del self.surname


jane = Person('Jane', 'Smith')
print(jane.fullname)        # Access like attribute, no ()

jane.fullname = 'Jane Doe'  # Uses setter
print(jane.fullname)
print(jane.surname)         # 'Doe'

del jane.fullname           # Uses deleter
print(hasattr(jane, 'name'))

---
## 4. Magic / Dunder Methods

**Dunder** (double-underscore) methods let you define how your objects respond to built-in Python operations.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):                   # str(v), print(v)
        return f'Vector({self.x}, {self.y})'

    def __repr__(self):                  # Developer-friendly representation
        return f'Vector(x={self.x}, y={self.y})'

    def __add__(self, other):            # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):            # v1 - v2
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):           # v * 3
        return Vector(self.x * scalar, self.y * scalar)

    def __eq__(self, other):             # v1 == v2
        return self.x == other.x and self.y == other.y

    def __len__(self):                   # len(v)
        import math
        return int(math.sqrt(self.x**2 + self.y**2))


v1 = Vector(2, 3)
v2 = Vector(1, 4)

print(str(v1))        # __str__
print(repr(v1))       # __repr__
print(v1 + v2)        # __add__
print(v1 - v2)        # __sub__
print(v1 * 3)         # __mul__
print(v1 == v2)       # __eq__
print(v1 == Vector(2,3))  # True
print(len(v1))        # __len__

In [ ]:
# Common dunder methods overview
dunders = {
    '__init__':     'Constructor',
    '__del__':      'Destructor',
    '__str__':      'str() and print()',
    '__repr__':     'Developer representation',
    '__len__':      'len()',
    '__getitem__':  'obj[key]',
    '__setitem__':  'obj[key] = value',
    '__contains__': 'item in obj',
    '__add__':      'obj + other',
    '__eq__':       'obj == other',
    '__lt__':       'obj < other',
    '__call__':     'obj()',
    '__enter__':    'with obj:',
    '__exit__':     'end of with block',
}
for name, desc in dunders.items():
    print(f'{name:20} → {desc}')

---
## 5. Multiple Inheritance

Python supports inheriting from **multiple parent classes**. This is powerful but use it carefully.

In [ ]:
class Flyable:
    def fly(self):
        return f'{self.name} is flying'

class Swimmable:
    def swim(self):
        return f'{self.name} is swimming'

class Person:
    def __init__(self, name):
        self.name = name
    def talk(self):
        return f'{self.name} is talking'


# Duck inherits from ALL THREE
class Duck(Flyable, Swimmable, Person):
    def quack(self):
        return 'Quack!'


donald = Duck('Donald')
print(donald.fly())    # from Flyable
print(donald.swim())   # from Swimmable
print(donald.talk())   # from Person
print(donald.quack())  # from Duck

In [ ]:
# MRO — Method Resolution Order
# Python follows the MRO to decide which class's method wins
print(Duck.__mro__)

---
## 6. Type Annotations (Hints)

Type hints make code more readable and allow static checkers (mypy) to catch bugs early.  
Python does **not enforce** them at runtime — they are just hints.

In [ ]:
def add(x: int, y: int) -> int:
    return x + y

def greet(name: str) -> str:
    return f'Hello, {name}!'

print(add(3, 5))
print(greet('Alice'))

# Annotations are stored in __annotations__
print(add.__annotations__)
print(greet.__annotations__)

In [ ]:
from typing import List, Optional, Dict

def find_max(numbers: List[int]) -> Optional[int]:
    """Returns max or None if list is empty."""
    return max(numbers) if numbers else None

print(find_max([3, 1, 9, 4]))  # 9
print(find_max([]))             # None

---
##  Quick Summary

| Concept | Key Point |
|---------|----------|
| **Polymorphism** | Same method name, different behavior per class |
| **Abstract Class** | Define a contract; subclasses must implement abstract methods |
| `@classmethod` | Bound to class (`cls`); use for factory methods |
| `@staticmethod` | Not bound to class or instance; pure utility |
| `@property` | Access method like an attribute (no `()` needed) |
| **Dunder methods** | Hook into Python's built-in operations (`+`, `len`, `str`, etc.) |
| **Multiple inheritance** | Inherit from multiple classes; Python uses MRO to resolve conflicts |
| **Type hints** | Improve readability; not enforced at runtime |

---

###  OOP in One Picture

```
                     Shape (abstract)
                    /               \
               Rectangle          Circle
               .area()            .area()
               .perimeter()       .perimeter()
               .describe()  ← inherited from Shape

         Polymorphism:
         for shape in [Rectangle(4,5), Circle(7)]:
             print(shape.describe())   # works for both!
```